In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# QESEM: O Funcție Qiskit de Qedma
*Consultă [referința API](https://docs.quantum.ibm.com/api/functions/qedma-qesem)*

> **Note:** Funcțiile Qiskit sunt o funcționalitate experimentală disponibilă doar utilizatorilor din planurile IBM Quantum&reg; Premium, Flex și On-Prem (prin API-ul IBM Quantum Platform). Acestea se află în stadiu de previzualizare și pot suferi modificări.

## Prezentare generală
Deși unitățile de procesare cuantică s-au îmbunătățit considerabil în ultimii ani, erorile cauzate de zgomot și imperfecțiunile hardware-ului existent rămân o provocare centrală pentru dezvoltatorii de algoritmi cuantici. Pe măsură ce domeniul se apropie de calcule cuantice la scară utilitară care nu pot fi verificate clasic, soluțiile pentru eliminarea zgomotului cu acuratețe garantată devin din ce în ce mai importante. Pentru a depăși această provocare, Qedma a dezvoltat Quantum Error Mitigation (QESEM), integrat fără întreruperi pe IBM Quantum Platform ca [Funcție Qiskit](/guides/functions).

Cu QESEM, utilizatorii pot rula circuitele lor cuantice pe QPU-uri cu zgomot pentru a obține rezultate precise, fără erori, cu costuri de timp QPU extrem de eficiente, aproape de limitele fundamentale. Pentru a realiza acest lucru, QESEM utilizează o suită de metode proprietare dezvoltate de Qedma pentru caracterizarea și reducerea erorilor. Tehnicile de reducere a erorilor includ optimizarea porților, transpilarea conștientă de zgomot, suprimarea erorilor (ES) și atenuarea nebiasată a erorilor (EM). Prin această combinație de metode bazate pe caracterizare, utilizatorii pot obține rezultate fiabile, fără erori, pentru circuite cuantice generice de volum mare, deblocând aplicații care nu ar putea fi realizate altfel.

Pentru o descriere completă a componentelor de bază, precum și o demonstrație la scară utilitară, consultați articolul [Reliable high-accuracy error mitigation for utility-scale quantum circuits](https://arxiv.org/abs/2508.10997).
## Descriere
Poți folosi funcția QESEM de Qedma pentru a estima și executa cu ușurință circuitele tale cu suprimare și atenuare a erorilor, obținând volume de circuit mai mari și acuratețe superioară. Pentru a utiliza QESEM, furnizezi un circuit cuantic, un set de observabile de măsurat, o acuratețe statistică țintă pentru fiecare observabilă și un QPU ales. Înainte de a rula circuitul la acuratețea țintă, poți estima timpul QPU necesar pe baza unui calcul analitic care nu necesită execuția circuitului. Odată ce ești mulțumit de estimarea timpului QPU, poți executa circuitul cu QESEM.

Când execuți un circuit, QESEM rulează un protocol de caracterizare a dispozitivului adaptat circuitului tău, producând un model de zgomot fiabil pentru erorile care apar în circuit. Pe baza caracterizării, QESEM implementează mai întâi transpilarea conștientă de zgomot pentru a mapa circuitul de intrare pe un set de qubiți și porți fizice, minimizând zgomotul care afectează observabila țintă. Acestea includ porțile disponibile nativ (CX/CZ pe dispozitivele IBM&reg;), precum și porți suplimentare optimizate de QESEM, formând setul extins de porți al QESEM. QESEM rulează apoi un set de circuite ES și EM bazate pe caracterizare pe QPU și colectează rezultatele măsurătorilor acestora. Acestea sunt apoi post-procesate clasic pentru a furniza o valoare de așteptare nebiasată și o marjă de eroare pentru fiecare observabilă, corespunzătoare acurateței solicitate.

![Prezentare generală Qedma QESEM](../docs/images/guides/qedma-qesem/overview.svg)
S-a demonstrat că QESEM oferă rezultate cu acuratețe ridicată pentru o varietate de aplicații cuantice și pe cele mai mari volume de circuit realizabile astăzi. QESEM oferă următoarele funcționalități orientate către utilizator, demonstrate în secțiunea de benchmark-uri de mai jos:
-	**Acuratețe garantată:** QESEM produce estimări nebiasate pentru valorile de așteptare ale observabilelor. Metoda sa EM este echipată cu garanții teoretice care — împreună cu caracterizarea de ultimă generație a Qedma — asigură că atenuarea converge către ieșirea circuitului fără zgomot, cu o acuratețe specificată de utilizator. Spre deosebire de multe metode EM euristice predispuse la erori sistematice sau biasuri, acuratețea garantată a QESEM este esențială pentru asigurarea unor rezultate fiabile în circuite și observabile cuantice generice.
-	**Scalabilitate pe QPU-uri mari:** Timpul QPU al QESEM depinde de volumele circuitelor, dar este altfel independent de numărul de qubiți. Qedma a demonstrat QESEM pe cele mai mari dispozitive cuantice disponibile astăzi, inclusiv dispozitivele IBM Quantum Eagle de 127 qubiți și Heron de 133 qubiți.
-	**Independent de aplicație:** QESEM a fost demonstrat pe o varietate de aplicații, inclusiv simulare Hamiltoniană, VQE, QAOA și estimare de amplitudine. Utilizatorii pot introduce orice circuit cuantic și observabilă de măsurat și pot obține rezultate precise, fără erori. Singurele limitări sunt dictate de specificațiile hardware-ului și de timpul QPU alocat, care determină volumele de circuit accesibile și acuratețele ieșirii. Spre deosebire de aceasta, multe soluții de reducere a erorilor sunt specifice aplicației sau implică euristici necontrolate, ceea ce le face inaplicabile pentru circuite și aplicații cuantice generice.
-  **Set extins de porți:** QESEM suportă porți cu unghiuri fracționare și furnizează porți $Rzz(\theta)$ cu unghiuri fracționare optimizate de Qedma pe dispozitivele IBM Quantum Heron și Eagle. Acest set extins de porți permite o compilare mai eficientă și deblochează volume de circuit mai mari cu un factor de până la 2 față de compilarea implicită CX/CZ.
-	**Observabile multibase:** QESEM suportă observabile de intrare compuse din mai multe șiruri Pauli necomutante, cum ar fi Hamiltonieni generici. Alegerea bazelor de măsurare și optimizarea alocării resurselor QPU (shot-uri și circuite) sunt apoi efectuate automat de QESEM pentru a minimiza timpul QPU necesar la acuratețea solicitată. Această optimizare, care ține cont de fidelitățile hardware-ului și de ratele de execuție, îți permite să rulezi circuite mai profunde și să obții acuratețe mai mare.
## Benchmark-uri
QESEM a fost testat pe o gamă largă de cazuri de utilizare și aplicații. Următoarele exemple te pot ajuta să evaluezi ce tipuri de sarcini de lucru poți rula cu QESEM.

O figură de merit cheie pentru cuantificarea dificultății atât a atenuării erorilor, cât și a simulării clasice pentru un circuit și o observabilă dată este **volumul activ**: numărul de porți CNOT care afectează observabila în circuit. Volumul activ depinde de adâncimea și lățimea circuitului, de greutatea observabilei și de structura circuitului, care determină conul de lumină al observabilei. Pentru detalii suplimentare, consultați prezentarea de la [IBM Quantum Summit 2024](https://www.youtube.com/watch?v=Hd-IGvuARfE&t=1730s). QESEM oferă valoare deosebit de mare în regimul cu volum mare, furnizând rezultate fiabile pentru circuite și observabile generice.

![Volum activ](../docs/images/guides/qedma-qesem/active_volume.svg)

| Aplicație    | Număr de qubiți | Dispozitiv | Descrierea circuitului | Acuratețe | Timp total | Utilizare runtime |
| ---------  | ---------------- | ----- | -------------------------- | -------- | ---------- | ------------- |
| Circuit VQE  | 8              | Eagle (r3) | 21 straturi totale, 9 baze de măsurare, lanț 1D                    | 98%      | 35 min      | 14 min         |
| Kicked Ising   | 28              | Eagle (r3) | 3 straturi unice x 3 pași, topologie 2D heavy-hex                      | 97%     | 22 min      | 4 min          |
| Kicked Ising   | 28              | Eagle (r3) | 3 straturi unice x 8 pași, topologie 2D heavy-hex                     | 97%      | 116 min      | 23 min          |
| Simulare Hamiltoniană Trotterizată   | 40  | Eagle (r3)            | 2 straturi unice x 10 pași Trotter, lanț 1D                    | 97%      | 3 ore     | 25 min         |
| Simulare Hamiltoniană Trotterizată   | 119  | Eagle (r3)           | 3 straturi unice x 9 pași Trotter, topologie 2D heavy-hex                    | 95%      | 6,5 ore     | 45 min         |
| Kicked Ising  | 136             | Heron (r2) | 3 straturi unice x 15 pași, topologie 2D heavy-hex                    | 99%      | 52 min             | 9 min           |

Acuratețea este măsurată aici relativ la valoarea ideală a observabilei: $\frac{\langle O \rangle_{ideal} - \epsilon}{\langle O \rangle_{ideal}}$, unde '$\epsilon$' este precizia absolută a atenuării (stabilită de intrarea utilizatorului), iar $\langle O \rangle_{ideal}$ este observabila la circuitul fără zgomot.
„Utilizarea runtime" măsoară utilizarea benchmark-ului în modul batch (suma utilizărilor job-urilor individuale), în timp ce „timpul total" măsoară utilizarea în modul sesiune (timpul real al experimentului), care include timpii clasici și de comunicație suplimentari. QESEM este disponibil pentru execuție în ambele moduri, astfel încât utilizatorii să poată face cel mai bun uz al resurselor disponibile.

Circuitele Kicked Ising de 28 qubiți simulează Cvazicristalul în Timp Discret studiat de Shinjo et al. (vezi [arXiv 2403.16718](https://arxiv.org/abs/2403.16718) și [Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)) pe trei bucle conectate ale ibm_kawasaki. Parametrii circuitului folosiți aici sunt $(\theta_x, \theta_z) = (0.9 \pi, 0)$, cu o stare inițială feromagnetică $| \psi_0 \rangle = | 0 \rangle ^{\otimes n}$. Observabila măsurată este valoarea absolută a magnetizării $M = |\frac{1}{28} \sum_{i=0}^{27} \langle Z_i \rangle|$. Experimentul Kicked Ising la scară utilitară a fost rulat pe cei mai buni 136 qubiți ai ibm_fez; acest benchmark particular a fost rulat la unghiul Clifford $(\theta_x, \theta_z) = (\pi, 0)$, la care volumul activ crește lent cu adâncimea circuitului, ceea ce — împreună cu fidelitățile ridicate ale dispozitivului — permite acuratețe ridicată la un timp de execuție scurt.

Circuitele de simulare Hamiltoniană Trotterizată sunt pentru un model Ising cu câmp transversal la unghiuri fracționare: $(\theta_{zz}, \theta_x) = (\pi / 4, \pi /8)$ și respectiv $(\theta_{zz}, \theta_x) = (\pi / 6, \pi / 8)$ (vezi [Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)). Circuitul la scară utilitară a fost rulat pe cei mai buni 119 qubiți ai ibm_brisbane, în timp ce experimentul de 40 qubiți a fost rulat pe cel mai bun lanț disponibil. Acuratețea este raportată pentru magnetizare; rezultate cu acuratețe ridicată au fost obținute și pentru observabile de greutate mai mare.

Circuitul VQE a fost dezvoltat împreună cu cercetători de la Centrul pentru Tehnologie și Aplicații Cuantice de la Deutsches Elektronen-Synchrotron (DESY). Observabila țintă a fost un Hamiltonian constând dintr-un număr mare de șiruri Pauli necomutante, subliniind performanța optimizată a QESEM pentru observabile multi-bază. Atenuarea a fost aplicată unui ansatz optimizat clasic; deși aceste rezultate sunt încă nepublicate, rezultate de aceeași calitate vor fi obținute pentru diferite circuite cu proprietăți structurale similare.
## Primii pași
Autentifică-te folosind [cheia ta API IBM Quantum Platform](http://quantum.cloud.ibm.com/) și selectează Funcția Qiskit QESEM după cum urmează. (Acest fragment de cod presupune că ți-ai [salvat deja contul](/guides/functions#install-qiskit-functions-catalog-client) în mediul local.)

In [1]:
import qiskit

from qiskit_ibm_catalog import QiskitFunctionsCatalog


catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# verify that you have access to the function
catalog.list()

[QiskitFunction(qunova/hivqe-chemistry),
 QiskitFunction(global-data-quantum/quantum-portfolio-optimizer),
 QiskitFunction(algorithmiq/tem),
 QiskitFunction(qedma/qesem),
 QiskitFunction(multiverse/singularity),
 QiskitFunction(ibm/circuit-function),
 QiskitFunction(q-ctrl/optimization-solver),
 QiskitFunction(colibritd/quick-pde),
 QiskitFunction(q-ctrl/performance-management),
 QiskitFunction(kipu-quantum/iskay-quantum-optimizer)]

In [2]:
# load the function
qesem_function = catalog.load("qedma/qesem")

## Examples

To get started, try this basic example of estimating the required QPU time to run QESEM for a given `pub`:

In [3]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy().name

In [4]:
circ = qiskit.QuantumCircuit(5)
circ.cx(0, 1)
circ.cx(2, 3)
circ.cx(1, 2)
circ.cx(3, 4)

avg_magnetization = qiskit.quantum_info.SparsePauliOp.from_sparse_list(
    [("Z", [q], 1 / 5) for q in range(5)], num_qubits=5
)
other_observable = qiskit.quantum_info.SparsePauliOp.from_sparse_list(
    [("ZZ", [0, 1], 1.0), ("XZ", [1, 4], 0.5)], num_qubits=5
)

time_estimation_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    options={
        "estimate_time_only": "analytical",
    },
    backend_name=backend_name,  # E.g. "ibm_fez"
)

time_estimate_result = time_estimation_job.result()

Poți folosi API-urile familiare Qiskit Serverless pentru a verifica starea sarcinii de lucru a Funcției tale Qiskit sau pentru a returna rezultatele:

In [5]:
sample_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    backend_name=backend_name,  # E.g. "ibm_fez"
)

Următorul fragment de cod descrie cum să recuperezi estimarea timpului QPU (când `estimate_time_only` este setat):

In [6]:
# Print the ID so you can use it later, if necessary
print(sample_job.job_id)

print(sample_job.status())
result = sample_job.result()

9a87a23f-82f5-429e-91fb-cc8a9d107980
QUEUED


Următorul fragment de cod demonstrează cum să recuperezi rezultatele atenuării (când `estimate_time_only` nu este setat) și metricile de execuție. Acestea conțin date esențiale care permit o înțelegere mai profundă a modului în care diferiți parametri influențează execuția QESEM. Pot fi relevante și la redactarea unui articol bazat pe cercetarea ta.

In [7]:
print(
    f"The estimated QPU time for this PUB is: "
    f"\n{time_estimate_result[0].metadata}"
)

The estimated QPU time for this PUB is: 
{'time_estimation_sec': 1800, 'description': None, 'instance': 'crn:v1:bluemix:public:quantum-computing:us-east:a/6c63dae5281147f1a0449b36e0aaba3a:ae40ab55-8c55-4042-9204-71a6541d56ec::', 'transpilation_level': 'standard', 'parallel_execution': False, 'total_qpu_time': 0, 'empirical_estimation_mitigation_results': None, 'resource_usage': {'RUNNING: MAPPING': {'CPU_TIME': 42.44837867887691, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: OPTIMIZING_FOR_HARDWARE': {'CPU_TIME': 17.879877626895905, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: WAITING_FOR_QPU': {'CPU_TIME': 0.0, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: EXECUTING_QPU': {'CPU_TIME': 0.0, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 0.0, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}}}


## Obținerea mesajelor de eroare
Dacă starea workload-ului tău este ERROR, folosește `job.result()` pentru a obține mesajul de eroare, astfel:

In [9]:
results = result[0]
print(f"Mitigated expectation values: \n{results.data.evs}")
print(f"Mitigated error-bar: \n{results.data.stds}")
noisy_results = results.metadata["noisy_results"]
print(f"Noisy expectation values: \n{noisy_results.evs}")
print(f"Noisy error-bar: \n{noisy_results.stds}")
print(f"Total QPU time: \n {results.metadata['total_qpu_time']}")
print(
    f"Gates fidelity measured during the experiment: "
    f"\n {results.metadata['gate_fidelities']}"
)
print(
    f"Total shots / mitigation shots: \n "
    f"{results.metadata['total_shots']} / "
    f"{results.metadata['mitigation_shots']}"
)
print("Transpiled circuits:")
for i, circuit in enumerate(results.metadata["transpiled_circs"]):
    print(f"Circuit {i}:")
    print(f"  Circuit: \n {circuit['circuit']}")
    if "qubit_map" in circuit:
        print(f"  Qubit mapping: \n {circuit['qubit_map']}")
    print(f"  Measurement bases: \n {circuit['num_measurement_bases']}")

Mitigated expectation values: 
[1.00169764 1.00460812]
Mitigated error-bar: 
[0.00155021 0.0099558 ]
Noisy expectation values: 
[0.95717143 0.94271429]
Noisy error-bar: 
[0.00206982 0.00872689]
Total QPU time: 
 150.0
Gates fidelity measured during the experiment: 
 {'CZ': 0.9979051606662408, 'ID1Q': 0.9993865847216725}
Total shots / mitigation shots: 
 495600 / 220400
Transpiled circuits:
Circuit 0:
  Circuit: 
 OPENQASM 3.0;
include "stdgates.inc";
bit[145] c0;
qubit[145] q0;
rz(-pi) q0[143];
rz(0) q0[141];
rz(-pi) q0[140];
sx q0[143];
sx q0[141];
sx q0[140];
rz(-pi/2) q0[143];
rz(pi/2) q0[141];
rz(-pi/2) q0[140];
sx q0[143];
sx q0[141];
sx q0[140];
rz(pi/2) q0[143];
rz(-pi/2) q0[141];
rz(pi/2) q0[140];
barrier q0[140], q0[141], q0[142], q0[143], q0[144];
cz q0[144], q0[143];
cz q0[142], q0[141];
barrier q0[144], q0[143], q0[142], q0[141];
barrier q0[140], q0[141], q0[142], q0[143], q0[144];
rz(-pi) q0[144];
rz(-pi/2) q0[143];
rz(0) q0[142];
rz(-pi/2) q0[141];
sx q0[144];
sx q0[143];

## Fetch error messages

If your workload status is ERROR, use `job.result()` to fetch the error message to fetch the error message as follows:

In [14]:
# Get the result and truncate for readability
result = sample_job.result()
result_str = str(result)
max_length = 500  # Adjust this value as necessary

if len(result_str) > max_length:
    truncated = (
        result_str[:max_length]
        + f"... (truncated {len(result_str) - max_length} characters)"
    )
else:
    truncated = result_str

print(truncated)

PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(2,), dtype=float64>), stds=np.ndarray(<shape=(2,), dtype=float64>), shape=(2,)), metadata={'gate_fidelities': {'CZ': 0.9979051606662408, 'ID1Q': 0.9993865847216725}, 'total_shots': 495600, 'mitigation_shots': 220400, 'transpiled_circs': [{'circuit': 'OPENQASM 3.0;\ninclude "stdgates.inc";\nbit[145] c0;\nqubit[145] q0;\nrz(-pi) q0[143];\nrz(0) q0[141];\nrz(-pi) q0[140];\nsx q0[143];\nsx q0[141];\nsx q0[140];\nrz(-pi/2) q0[143];\nrz(pi... (truncated 4174 characters)


## Obținerea suportului

Echipa de suport Qedma este aici să te ajute! Dacă întâmpini probleme sau ai întrebări despre utilizarea funcției QESEM Qiskit, nu ezita să ne contactezi. Personalul nostru de suport cu experiență și prietenos este pregătit să te asiste cu orice problemă tehnică sau întrebare ai.

Ne poți trimite un e-mail la support@qedma.com pentru asistență. Te rog să incluzi cât mai multe detalii posibil despre problema întâmpinată, pentru a ne ajuta să oferim un răspuns rapid și precis. Poți contacta, de asemenea, reprezentantul tău dedicat Qedma POC prin e-mail sau telefon.

Pentru a ne ajuta să te asistăm mai eficient, te rog să furnizezi următoarele informații când ne contactezi:

- O descriere detaliată a problemei
- ID-ul job-ului
- Orice mesaje de eroare sau coduri relevante

Ne angajăm să îți oferim suport prompt și eficient pentru a te asigura că ai cea mai bună experiență posibilă cu Funcția noastră Qiskit.

Suntem mereu în căutarea de modalități de a îmbunătăți produsul nostru și îți primim sugestiile cu drag! Dacă ai idei despre cum putem îmbunătăți serviciile noastre sau funcționalități pe care ai dori să le vedem, te rog să ne trimiți gândurile tale la support@qedma.com.

## Pașii următori

> **Tip:** - [Solicită acces la Qedma QESEM](https://quantum.cloud.ibm.com/functions?id=qedma-qesem).
> - Vizitează [referința API](https://docs.quantum.ibm.com/api/functions/qedma-qesem) pentru această Funcție Qiskit.
> - Consultă [Bauman, N. P., et al. (2025). Coupled Cluster Downfolding Theory in Simulations of Chemical Systems on Quantum Hardware. arXiv preprint arXiv:2507.01199](https://arxiv.org/pdf/2507.01199).
> - Consultă [Aharonov, D., et al. (2025). Reliable high-accuracy error mitigation for utility-scale quantum circuits. arXiv preprint arXiv:2508.10997](https://arxiv.org/pdf/2508.10997).